# Honest holdout test set

Επιλέγουμε 5 HC + 5 PD subjects από κάθε dataset, σύνολο 30 holdout subjects. Επανεκπαιδεύουμε τα μοντέλα χωρίς αυτά. Όταν δοκιμάζουμε τα συγκεκριμένα δείγματα στην εφαρμογή, λαμβάνουμε τίμιο generalization prediction, καθώς το μοντέλο δεν τα έχει δει στην εκπαίδευση.

In [ ]:
import sys, joblib, json
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline

from src.features import FEATURE_NAMES

MODELS = Path('../models')
DATA = Path('../data')
rng = np.random.default_rng(2024)  # fixed seed για reproducibility

## Επιλογή holdout subjects από κάθε dataset

In [ ]:
holdout = {'uci': {'hc': [], 'pd': []},
           'iyer': {'hc': [], 'pd': []},
           'mdvr': {'hc': [], 'pd': []}}

# UCI: 5 HC + 5 PD subjects
uci = pd.read_csv(DATA / 'uci/pd_speech_features.csv', header=1)
uci_hc = uci[uci['class']==0]['id'].unique()
uci_pd = uci[uci['class']==1]['id'].unique()
holdout['uci']['hc'] = sorted(rng.choice(uci_hc, 5, replace=False).tolist())
holdout['uci']['pd'] = sorted(rng.choice(uci_pd, 5, replace=False).tolist())

# Iyer: 5 HC + 5 PD subjects (each subject = 1 file)
iyer = pd.read_csv(DATA / 'iyer/iyer_features_8khz.csv')
iyer_hc = iyer[iyer['class']==0]['subject'].unique()
iyer_pd = iyer[iyer['class']==1]['subject'].unique()
holdout['iyer']['hc'] = sorted(rng.choice(iyer_hc, 5, replace=False).tolist())
holdout['iyer']['pd'] = sorted(rng.choice(iyer_pd, 5, replace=False).tolist())

# MDVR: 5 HC + 5 PD subjects
mdvr = pd.read_csv(DATA / 'mdvr_kcl/mdvr_features.csv')
mdvr_hc = mdvr[mdvr['class']==0]['subject'].unique()
mdvr_pd = mdvr[mdvr['class']==1]['subject'].unique()
holdout['mdvr']['hc'] = sorted(rng.choice(mdvr_hc, 5, replace=False).tolist())
holdout['mdvr']['pd'] = sorted(rng.choice(mdvr_pd, 5, replace=False).tolist())

print('=== HOLDOUT SUBJECTS ===\n')
for ds, classes in holdout.items():
    print(f'{ds.upper()}:')
    print(f'  HC: {classes["hc"]}')
    print(f'  PD: {classes["pd"]}')
    print()

# Save holdout list
with open(MODELS / 'holdout_subjects.json', 'w') as f:
    json.dump(holdout, f, indent=2, default=str)

## Retrain χωρίς τα holdout subjects

In [ ]:
def train_pair(X, y):
    rf = Pipeline([('scaler', RobustScaler()), 
                   ('clf', RandomForestClassifier(n_estimators=300, random_state=42,
                                                  n_jobs=-1, class_weight='balanced'))])
    svm = Pipeline([('scaler', RobustScaler()),
                    ('clf', SVC(kernel='rbf', C=1.0, probability=True,
                                class_weight='balanced', random_state=42))])
    rf.fit(X, y)
    svm.fit(X, y)
    return rf, svm

In [ ]:
# Iyer: εξαιρώ holdout subjects
all_holdout_iyer = holdout['iyer']['hc'] + holdout['iyer']['pd']
iyer_train = iyer[~iyer['subject'].isin(all_holdout_iyer)].reset_index(drop=True)
print(f'Iyer train: {len(iyer_train)} samples (από {len(iyer)})')

rf, svm = train_pair(iyer_train[FEATURE_NAMES], iyer_train['class'].values)
joblib.dump(rf, MODELS / 'iyer_8khz.joblib')
joblib.dump(svm, MODELS / 'iyer_8khz_svm.joblib')
print('Iyer models retrained')

In [ ]:
# MDVR: εξαιρώ holdout subjects
all_holdout_mdvr = holdout['mdvr']['hc'] + holdout['mdvr']['pd']
mdvr_train = mdvr[~mdvr['subject'].isin(all_holdout_mdvr)].reset_index(drop=True)
print(f'MDVR train: {len(mdvr_train)} samples (από {len(mdvr)})')

rf, svm = train_pair(mdvr_train[FEATURE_NAMES], mdvr_train['class'].values)
joblib.dump(rf, MODELS / 'mdvr_model.joblib')
joblib.dump(svm, MODELS / 'mdvr_svm.joblib')
print('MDVR models retrained')

In [ ]:
# UCI: balanced + εξαιρώ holdout
all_holdout_uci = holdout['uci']['hc'] + holdout['uci']['pd']
clean_features = joblib.load(MODELS / 'uci_clean_features.joblib')

uci_train_pool = uci[~uci['id'].isin(all_holdout_uci)].reset_index(drop=True)
rng2 = np.random.default_rng(42)
hc_subj = uci_train_pool[uci_train_pool['class']==0]['id'].unique()  # 64-5=59
pd_subj_pool = uci_train_pool[uci_train_pool['class']==1]['id'].unique()  # 188-5=183
pd_subj = rng2.choice(pd_subj_pool, len(hc_subj), replace=False)
uci_train = uci_train_pool[uci_train_pool['id'].isin(np.concatenate([hc_subj, pd_subj]))].reset_index(drop=True)
print(f'UCI train: {len(uci_train)} samples (από {len(uci)}, balanced)')

rf, svm = train_pair(uci_train[clean_features], uci_train['class'].values)
joblib.dump(rf, MODELS / 'uci_balanced_noint.joblib')
joblib.dump(svm, MODELS / 'uci_balanced_noint_svm.joblib')
print('UCI models retrained')

## Quick predictions στα holdout samples (sanity check)

In [ ]:
iyer_holdout = iyer[iyer['subject'].isin(all_holdout_iyer)]
iyer_rf = joblib.load(MODELS / 'iyer_8khz.joblib')
iyer_svm = joblib.load(MODELS / 'iyer_8khz_svm.joblib')

print('=== Iyer holdout predictions ===')
print(f'{"Subject":<60} {"True":<6} {"RF":<6} {"SVM":<6} {"Ens":<6}')
correct = 0
for _, row in iyer_holdout.iterrows():
    x = pd.DataFrame([row[FEATURE_NAMES].values], columns=FEATURE_NAMES)
    rf_p = iyer_rf.predict_proba(x)[0, 1]
    svm_p = iyer_svm.predict_proba(x)[0, 1]
    ens = (rf_p + svm_p) / 2
    pred = 'PD' if ens >= 0.5 else 'HC'
    true = 'PD' if row['class'] == 1 else 'HC'
    if pred == true: correct += 1
    sub = row['subject'][:55]
    print(f'{sub:<60} {true:<6} {rf_p:.2f}  {svm_p:.2f}  {ens:.2f} -> {pred}')
print(f'\nIyer holdout accuracy: {correct}/{len(iyer_holdout)} = {correct/len(iyer_holdout)*100:.1f}%')

In [ ]:
mdvr_holdout = mdvr[mdvr['subject'].isin(all_holdout_mdvr)]
mdvr_rf = joblib.load(MODELS / 'mdvr_model.joblib')
mdvr_svm = joblib.load(MODELS / 'mdvr_svm.joblib')

print('=== MDVR holdout predictions ===')
print(f'{"Subject":<25} {"File":<35} {"True":<6} {"RF":<6} {"SVM":<6} {"Ens":<6}')
correct = 0
for _, row in mdvr_holdout.iterrows():
    x = pd.DataFrame([row[FEATURE_NAMES].values], columns=FEATURE_NAMES)
    rf_p = mdvr_rf.predict_proba(x)[0, 1]
    svm_p = mdvr_svm.predict_proba(x)[0, 1]
    ens = (rf_p + svm_p) / 2
    pred = 'PD' if ens >= 0.5 else 'HC'
    true = 'PD' if row['class'] == 1 else 'HC'
    if pred == true: correct += 1
    print(f'{row["subject"]:<25} {row["filename"][:32]:<35} {true:<6} {rf_p:.2f}  {svm_p:.2f}  {ens:.2f} -> {pred}')
print(f'\nMDVR holdout accuracy: {correct}/{len(mdvr_holdout)} = {correct/len(mdvr_holdout)*100:.1f}%')

In [ ]:
uci_holdout = uci[uci['id'].isin(all_holdout_uci)]
uci_rf = joblib.load(MODELS / 'uci_balanced_noint.joblib')
uci_svm = joblib.load(MODELS / 'uci_balanced_noint_svm.joblib')

print('=== UCI holdout predictions ===')
print(f'{"Subject ID":<12} {"True":<6} {"RF":<6} {"SVM":<6} {"Ens":<6}')
correct = 0
for _, row in uci_holdout.iterrows():
    x = pd.DataFrame([row[clean_features].values], columns=clean_features)
    rf_p = uci_rf.predict_proba(x)[0, 1]
    svm_p = uci_svm.predict_proba(x)[0, 1]
    ens = (rf_p + svm_p) / 2
    pred = 'PD' if ens >= 0.5 else 'HC'
    true = 'PD' if row['class'] == 1 else 'HC'
    if pred == true: correct += 1
    print(f'{row["id"]:<12} {true:<6} {rf_p:.2f}  {svm_p:.2f}  {ens:.2f} -> {pred}')
print(f'\nUCI holdout accuracy: {correct}/{len(uci_holdout)} = {correct/len(uci_holdout)*100:.1f}%')